# UAE Rental Price Prediction — 01 Data Preparation

**Course:** CDS4003 — Data Science for Business Application  
**Goal:** Reproduce the shared Altair AI Studio / RapidMiner cleaning steps using Python.

**Dataset:** `data/UAE_properties.csv`  
**Output:** `data/UAE_properties_clean_python.csv` (keeps the existing RapidMiner output untouched)

This notebook loads and checks the original data, retains 11 relevant attributes, fills missing coordinates with their averages, removes invalid rental values, converts **City**, **Type**, and **Furnishing** into 0/1 columns, and preserves **Rent_category** for classification.

> **Important for future modelling:** These are the shared CLO2 cleaning steps. When we test machine-learning models, any learned preprocessing (for example, mean imputation and scaling) should be fitted on the training data only, inside validation folds, to prevent data leakage. Model evaluation comes later.

## 1. Import libraries and load the original dataset

The code looks for the dataset in common GitHub/Jupyter and Google Colab locations. When using Colab, upload `UAE_properties.csv` to the Colab files panel first.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

# Works in the repository, Jupyter Notebook, or Colab (when the CSV is uploaded).
possible_paths = [
    Path('../data/UAE_properties.csv'),
    Path('data/UAE_properties.csv'),
    Path('UAE_properties.csv'),
    Path('/content/UAE_properties.csv'),
]

source_path = next((p for p in possible_paths if p.is_file()), None)
if source_path is None:
    raise FileNotFoundError(
        'UAE_properties.csv not found. Open the notebook inside the GitHub project '
        'or upload UAE_properties.csv to the Colab files panel.'
    )

df = pd.read_csv(source_path, low_memory=False)
df.columns = df.columns.str.strip().str.replace('\ufeff', '', regex=False)
print('Loaded:', source_path)
print('Original rows:', len(df))
print('Original columns:', df.shape[1])
display(df.head())

## 2. Understand the data and check data quality

Check column names and data types, missing values, duplicated records, and invalid rental prices. We do not remove anything in this step.

In [ ]:
print('Column names and types:')
display(pd.DataFrame({'Attribute': df.columns, 'Data type': df.dtypes.astype(str).values}))

print('Missing values BEFORE cleaning:')
display(df.isna().sum().rename('Missing count').to_frame())

print('Total missing cells:', int(df.isna().sum().sum()))
print('Exact duplicate rows:', int(df.duplicated().sum()))

# Rent should be numerical so we can identify invalid zero/negative values.
df['Rent'] = pd.to_numeric(df['Rent'], errors='coerce')
print('Rent <= 0:', int(df['Rent'].le(0).sum()))
print('Missing/non-numeric Rent:', int(df['Rent'].isna().sum()))
print('Rent minimum:', df['Rent'].min())
print('Rent maximum:', df['Rent'].max())
if 'Area_in_sqft' in df:
    print('Area maximum:', pd.to_numeric(df['Area_in_sqft'], errors='coerce').max())

## 3. Keep the 11 useful attributes

The same columns were selected in RapidMiner. `Rent` is needed for regression; `Rent_category` is needed for classification. The other columns describe the property and its location.

In [ ]:
keep_columns = [
    'Rent', 'Rent_category', 'Beds', 'Baths', 'Type',
    'Area_in_sqft', 'City', 'Furnishing', 'Latitude',
    'Longitude', 'Age_of_listing_in_days'
]

missing_columns = [c for c in keep_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f'Required columns are missing from the CSV: {missing_columns}')

removed_columns = [c for c in df.columns if c not in keep_columns]
clean_df = df[keep_columns].copy()
print('Retained columns:', len(keep_columns), keep_columns)
print('Removed columns:', len(removed_columns), removed_columns)
print('Shape after selection:', clean_df.shape)

## 4. Remove exact duplicates (if any) and fill missing coordinates

Duplicate rows are checked and removed if found. Missing **Latitude** and **Longitude** values are replaced using the average of each column, as in our RapidMiner process. This retains more properties, but the replacement coordinates are not exact locations.

In [ ]:
before_duplicates = len(clean_df)
clean_df = clean_df.drop_duplicates().copy()
print('Exact duplicate rows removed:', before_duplicates - len(clean_df))

for col in ['Latitude', 'Longitude']:
    clean_df[col] = pd.to_numeric(clean_df[col], errors='coerce')
    missing_before = int(clean_df[col].isna().sum())
    average_value = clean_df[col].mean()
    if pd.isna(average_value):
        raise ValueError(f'Cannot fill {col}: all values are missing')
    clean_df[col] = clean_df[col].fillna(average_value)
    print(f'{col}: filled {missing_before} missing values with average {average_value:.5f}')

print('Missing cells after coordinate imputation:', int(clean_df.isna().sum().sum()))

## 5. Filter out invalid records

Keep only properties with **Rent > 0** and coordinates that are not missing. In the RapidMiner workflow, the missing-coordinate filter was applied *after* filling missing values, so it usually does not remove further rows.

In [ ]:
rows_before_filter = len(clean_df)
clean_df = clean_df[
    (clean_df['Rent'] > 0)
    & clean_df['Latitude'].notna()
    & clean_df['Longitude'].notna()
].copy()

print('Rows removed by filters:', rows_before_filter - len(clean_df))
print('Rows after filters:', len(clean_df))
print('Minimum Rent after filtering:', clean_df['Rent'].min())

## 6. Convert selected categorical columns to numbers (dummy coding)

Convert **City**, **Type**, and **Furnishing** into separate 0/1 columns. Keep `Rent_category` as Low / Medium / High. It is **not** encoded at this stage because it is the future classification target.

In [ ]:
categorical_inputs = ['City', 'Type', 'Furnishing']
clean_df = pd.get_dummies(
    clean_df,
    columns=categorical_inputs,
    prefix_sep=' = ',
    dtype=int,
    dummy_na=False,
)

print('Rows after encoding:', len(clean_df))
print('Columns after encoding:', clean_df.shape[1])
print('Rent_category preserved:', 'Rent_category' in clean_df.columns)
print('Rent_category values:', sorted(clean_df['Rent_category'].dropna().astype(str).unique().tolist()))
display(clean_df.head())

## 7. Check the prepared dataset (before vs. after)

Confirm the number of records, attributes, remaining missing values, and label column. These checks are useful for the project report and for comparing with RapidMiner results.

In [ ]:
check_table = pd.DataFrame({
    'Measure': ['Rows', 'Columns', 'Missing cells', 'Rent <= 0', 'Exact duplicate rows'],
    'Original': [
        len(df), len(df.columns), int(df.isna().sum().sum()),
        int(df['Rent'].le(0).sum()), int(df.duplicated().sum())
    ],
    'Prepared': [
        len(clean_df), clean_df.shape[1], int(clean_df.isna().sum().sum()),
        int(clean_df['Rent'].le(0).sum()), int(clean_df.duplicated().sum())
    ]
})
display(check_table)

expected_rows = 73727
expected_columns = 27
if len(df) == 73742:
    print('Report check — expected prepared rows:', expected_rows)
    print('Report check — expected prepared columns:', expected_columns)
    if len(clean_df) != expected_rows or clean_df.shape[1] != expected_columns:
        print('NOTE: Results differ from the RapidMiner report. Review unique categories and filtering.')
    else:
        print('Row/column counts match the RapidMiner report.')

assert 'Rent_category' in clean_df.columns, 'Classification label must be preserved'
assert clean_df['Rent'].gt(0).all(), 'All remaining rental prices must be positive'
assert clean_df[['Latitude', 'Longitude']].notna().all().all(), 'Coordinates must have no missing values'
print('Validation checks passed.')

## 8. Save the cleaned Python dataset

This creates a **separate** CSV file so the existing `UAE_properties_clean.csv` exported from RapidMiner is not overwritten. The generated CSV can be used in later notebooks, after any model-specific adjustments.

In [ ]:
# Save next to the original CSV, whether working in GitHub/Jupyter or Colab.
output_path = source_path.parent / 'UAE_properties_clean_python.csv'
clean_df.to_csv(output_path, index=False)

print('Saved:', output_path)
print('Final shape:', clean_df.shape)
print('File size (MB):', round(output_path.stat().st_size / (1024 * 1024), 2))

## Notes for the next stage

- **Regression:** Predict `Rent`. Exclude `Rent_category` from input features.
- **Classification:** Predict `Rent_category`. Exclude `Rent` and features derived from it.
- **Clustering:** No target label. Select features and scale them as needed.
- **Data quality limitation:** Average-imputed coordinates may not represent actual property locations. Extreme rent/area values still need investigation.
- **Model validation:** For unbiased evaluation, fit preprocessing on training folds rather than on the entire dataset.

**AI assistance:** Python notebook code was drafted with ChatGPT assistance. The team should inspect, run, and understand it, and declare the use of AI tools in the GitHub README as required by the rubric.